In [1]:
import math
import os
import sys
sys.path.append(os.path.abspath('.')) # to run files that are away
os.environ["WANDB_SILENT"] = "true"  # Suppress WandB logs

libraries = ["torch", "numpy", "polars"]
modules   = {lib: sys.modules.get(lib) for lib in libraries}

if not modules["torch"]:
    import torch
if not modules["numpy"]:
    import numpy as np
if not modules["polars"]:
    import polars as pl

import gc
import catboost as cb
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from load_and_rename_files import LogFilesProcessor, WaferFilesProcessor
from key_params import main_folder, NUM_WAFERS, dict_of_wafer_files, dict_of_log_files, step_col_name, COMMON_ID_COLS, COMMON_ID_COLS_MOD, parquet_folder_name

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:

df = pl.read_parquet(f'../ASM_data/{parquet_folder_name}/wafer_1_log.parquet')

from asm_data_wrangling import master_df, load_and_preprocess_wafer_data, load_and_process_log_files, \
    split_and_save_log_df_by_wafer, train_models


FileNotFoundError: No such file or directory (os error 2): ./ASM_data/2. marathon0/Wafer performance/Spatial property after step 4.csv

In [ ]:

# master_df, wafer_df_dict, y_df_dict, radius_wide_dict=load_and_preprocess_wafer_data(dict_of_wafer_files, main_folder, save=False)
# unique_marathon_runs_list = list(master_df["marathon_run"].unique())
# log_df = load_and_process_log_files(dict_of_log_files, log_processor, unique_marathon_runs_list, step_col_name, main_folder, save=False)
# split_and_save_log_df_by_wafer(log_df, NUM_WAFERS, main_folder, log_processor, overwrite = True)

master_df.head()



process time,marathon_run,step_id,#run,common signal_5_step1,common signal_6_step1,common signal_7_step1,common signal_34_step1,common signal_35_step1,common signal_36_step1,common signal_42_step1,common signal_46_step1,common signal_47_step1,common signal_48_step1,common signal_49_step1,common signal_50_step1,common signal_51_step1,common signal_52_step1,common signal_53_step1,common signal_54_step1,common signal_55_step1,common signal_56_step1,common signal_58_step1,common signal_59_step1,common signal_60_step1,common signal_62_step1,common signal_63_step1,common signal_66_step1,common signal_67_step1,common signal_69_step1,common signal_70_step1,common signal_71_step1,common signal_72_step1,common signal_74_step1,common signal_75_step1,common signal_76_step1,common signal_80_step1,…,rc1 signal_8_step1,rc1 signal_9_step1,rc1 signal_10_step1,rc1 signal_11_step1,rc1 signal_12_step1,rc1 signal_13_step1,rc1 signal_14_step1,rc1 signal_0_step2,rc1 signal_1_step2,rc1 signal_2_step2,rc1 signal_3_step2,rc1 signal_0_step3,rc1 signal_1_step3,rc1 signal_2_step3,rc1 signal_3_step3,rc1 signal_4_step3,rc1 signal_5_step3,rc1 signal_6_step3,rc1 signal_7_step3,rc1 signal_8_step3,rc1 signal_9_step3,rc1 signal_10_step3,rc1 signal_11_step3,rc1 signal_12_step3,rc1 signal_13_step3,rc1 signal_14_step3,rc1 signal_0_step4,rc1 signal_1_step4,rc1 signal_2_step4,rc1 signal_3_step4,rc1 signal_15_step1,rc1 signal_16_step1,rc1 signal_5_step2,rc1 signal_6_step2,rc1 signal_16_step3,rc1 signal_5_step4,rc1 signal_6_step4
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2024-08-02T14:30:42.976""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.2,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,2000.3,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,0.0,187.0,187.3,149.7,8.0,0.0,34.9,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""2024-08-02T14:30:43.026""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.1,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,2000.0,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,0.0,187.0,187.3,149.7,8.0,0.0,34.9,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""2024-08-02T14:30:43.076""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.1,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,1999.9,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,0.0,187.0,187.3,149.7,8.6,0.0,34.6,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""2024-08-02T14:30:43.126""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.1,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,2000.0,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,0.0,187.0,187.3,149.7,8.6,0.0,34.6,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""2024-08-02T14:30:43.176""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.1,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,1999.9,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,0.0,187.0,187.3,149.7,8.9,0.0,33.1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [ ]:
y_pred_cat.shape
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

type(y_pred_cat)

In [ ]:
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

plt.scatter(range(len(y_full_pd.iloc[1].values)), y_full_pd.iloc[1].values,
            label="True", facecolors='none', edgecolors='blue', s=8)
plt.scatter(range(len(y_pred_unscaled[1])), y_pred_unscaled[1],
            label="Predicted", facecolors='none', edgecolors='orange', s=8)
# plt.plot(y_full_pd.iloc[1].values, label="True")
# plt.plot(y_pred_unscaled[1], label="Predicted")
plt.legend()
plt.title("y_predicted vs y_actual")
plt.xlabel("Site ID (coordinate)")
plt.ylabel("Spatial property")
plt.show()


In [ ]:
# hyperparam search

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'estimator__num_leaves': [20, 31, 40, 50],
    'estimator__max_depth': [-1, 5, 10, 20],
    'estimator__min_data_in_leaf': [10, 20, 30],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__n_estimators': [100, 500, 1000]}

# model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', verbosity=0))
model = MultiOutputRegressor(LGBMRegressor(objective='regression', verbosity=-1))

search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV RMSE:", (-search.best_score_)**0.5)



##### Spatial data (M)

In [ ]:
def plot_wafer_property(df, property_col, title):
    plt.figure(figsize=(9, 5))
    for rc_value, group in df.group_by("RC"):
        x = group["#Run"].to_list()
        y = group[property_col].to_list()
        plt.scatter(x, y, label=f'RC {rc_value}', s=20)
    plt.xlabel("#Run")
    plt.ylabel(property_col)
    plt.title(title)
    plt.legend()
    plt.show()

plot_wafer_property(wafer_df, "Wafer property summary 1", "Wafer Property 1")
plot_wafer_property(wafer_df, "Wafer property summary 2", "Wafer Property 2")


##### Import timeseries data (S)

In [ ]:
"""Load data and make parquet files out of it"""

should_we_save_parquet_files = False

def _save_df_as_parquet_file(df: pl.dataframe, saving_location: str):
    df.write_parquet(saving_location)

def remove_unchanging_cols_from_df_and_save(df: pl.dataframe, col_name: str, should_we_save_parquet_files: bool) -> None:
    for run_id in df[col_name].unique().to_list():
        df_per_run    = df.filter(pl.col(col_name) == run_id)
        constant_cols = [col for col in df_per_run.columns
                     if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
                        #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
                         df_per_run.select(pl.col(col).n_unique()).item() == 1]
        df_per_run_filtered = df_per_run.drop(constant_cols)
        saving_location = f"{parquet_subfolder}/run_{run_id}.parquet"
        if should_we_save_parquet_files:
            _save_df_as_parquet_file(df_per_run_filtered, saving_location)

remove_unchanging_cols_from_df_and_save(log_df, "#Run", should_we_save_parquet_files)

# =============
# before making funcrtion:

# if should_we_save_parquet_files:
#     for run_id in log_df["#Run"].unique().to_list():
#         df_per_run = log_df.filter(pl.col("#Run") == run_id)
#         zero_cols  = [col for col in df_per_run.columns
#                      if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
#                         #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
#                          df_per_run.select(pl.col(col).n_unique()).item() == 1]
#         df_per_run_filtered = df_per_run.drop(zero_cols)
#         df_per_run_filtered.write_parquet(f"{parquet_subfolder}/run_{run_id}.parquet")


##### Timeseries data (S)

In [ ]:
run_number   = 15
parquet_file = f"./ASM_data/3. marathon1/Logs/split_by_run/run_{run_number}.parquet"
df           = pl.read_parquet(parquet_file)
df_pd        = df.to_pandas()
df_numeric   = df.select(pl.col(pl.NUMERIC_DTYPES))
X_np         = df_numeric.to_numpy()
X_scaled     = StandardScaler().fit_transform(X_np)

num_cols_to_plot = len(df_pd.columns)
num_rows         = math.ceil(math.sqrt(num_cols_to_plot))
num_cols_grid    = math.ceil(num_cols_to_plot / num_rows)

axes = df_pd.plot(subplots=True, figsize=(14, 12), layout=(num_rows, num_cols_grid), sharex=True, legend=False)

if isinstance(axes, np.ndarray):
    axes_flat = axes.flatten()
else:
    axes_flat = [axes]

column_names = df_pd.columns.tolist()

for i, ax in enumerate(axes_flat):
    if i < num_cols_to_plot: # Only set title for actual plots
        ax.set_title(column_names[i], fontsize='xx-small')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

for i in range(num_cols_to_plot, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout()
plt.suptitle(f'Run #{run_number} {log_file}', fontsize='large', y=1.02) # Adjust y to prevent overlap
plt.show()

In [ ]:
data_dimensionality = DimensionalityEstimator.estimate_dataset_dimensionality(df)
print(f"Recommended latent layer size: {data_dimensionality:.1f}")


In [ ]:
from sklearn.model_selection import train_test_split


X_train_np, X_test_np = train_test_split(X_scaled, test_size=0.2, random_state=42)
X_train               = torch.tensor(X_train_np, dtype=torch.float32)
X_test                = torch.tensor(X_test_np, dtype=torch.float32)
